<a href="https://colab.research.google.com/github/BryanGermano/exemplo-repo-front-1tdspy/blob/main/FoodSave_Pulse.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

FoodSave Pulse

In [ ]:


"""

             FoodSave Pulse
   Combatendo o desperdício, um prato por vez


Plataforma inteligente para redução do desperdício alimentar.
"""

import os
import sys
import time
import random
import datetime
import json
import math
import threading



class Cor:
    RESET   = "\033[0m"
    BOLD    = "\033[1m"
    DIM     = "\033[2m"

    VERDE       = "\033[32m"
    VERDE_CLARO = "\033[92m"
    AMARELO     = "\033[33m"
    AMARELO_CLARO = "\033[93m"
    AZUL        = "\033[34m"
    AZUL_CLARO  = "\033[94m"
    CIANO       = "\033[36m"
    CIANO_CLARO = "\033[96m"
    VERMELHO    = "\033[31m"
    MAGENTA     = "\033[35m"
    BRANCO      = "\033[97m"
    CINZA       = "\033[90m"

    BG_VERDE   = "\033[42m"
    BG_AZUL    = "\033[44m"
    BG_PRETO   = "\033[40m"

def colorir(texto, *estilos):
    return "".join(estilos) + texto + Cor.RESET

def limpar():
    os.system("cls" if os.name == "nt" else "clear")

def pausar(msg="  Pressione Enter para continuar..."):
    input(colorir(f"\n{msg}", Cor.CINZA))



class Banco:
    """Banco de dados em memória que simula dados reais da plataforma."""

    PRODUTOS_BASE = [
        ("Pão Francês",     "Padaria São Paulo",     "padaria",    2.50,  "pão"),
        ("Iogurte Natural", "Mercado Central",       "mercado",    4.80,  "laticínio"),
        ("Frango Grelhado", "Restaurante Bom Sabor", "restaurante",9.90,  "proteína"),
        ("Salada Mista",    "Café Orgânico",         "restaurante",6.50,  "vegetal"),
        ("Queijo Minas",    "Mini Mercado Família",  "mercado",    8.20,  "laticínio"),
        ("Macarrão ao Sugo","Cantina Italiana",      "restaurante",11.00, "massa"),
        ("Bolo de Cenoura", "Confeitaria Arte",      "padaria",    5.50,  "doce"),
        ("Leite Integral",  "Supermercado Econômico","mercado",    3.90,  "laticínio"),
        ("Arroz com Feijão","Restaurante Popular",   "restaurante",7.00,  "prato"),
        ("Croissant",       "Padaria Francesa",      "padaria",    3.20,  "pão"),
        ("Tomate Cereja",   "Hortifrúti Verde",      "mercado",    5.00,  "vegetal"),
        ("Lasanha",         "Cozinha Caseira",       "restaurante",13.50, "massa"),
    ]

    REGIOES = ["Centro", "Zona Norte", "Zona Sul", "Zona Leste", "Zona Oeste"]

    def __init__(self):
        self.usuarios = self._criar_usuarios()
        self.produtos  = self._criar_produtos()
        self.missoes   = self._criar_missoes()
        self.historico = self._criar_historico()
        self.doacoes   = self._criar_doacoes()
        self.usuario_atual = None

    def _criar_usuarios(self):
        return {
            "ana": {
                "nome": "Ana Lima", "email": "ana@email.com", "senha": "1234",
                "eco_points": 1_240, "resgates": 47, "nivel": "Guardião Verde ",
                "regiao": "Zona Sul", "tipo": "consumidor",
                "historico_pontos": [20, 35, 10, 50, 80, 45, 60, 90, 120, 95, 110, 85],
                "conquistas": ["Primeiro Resgate ", "10 Resgates ", "Mês Sustentável "],
                "meta_mensal": 200,
                "pontos_mes": 145,
            },
            "carlos": {
                "nome": "Carlos Melo", "email": "carlos@email.com", "senha": "5678",
                "eco_points": 890, "resgates": 31, "nivel": "Protetor Ambiental ",
                "regiao": "Centro", "tipo": "consumidor",
                "historico_pontos": [10, 20, 15, 40, 55, 30, 45, 70, 80, 60, 90, 75],
                "conquistas": ["Primeiro Resgate ", "10 Resgates "],
                "meta_mensal": 150,
                "pontos_mes": 98,
            },
        }

    def _criar_produtos(self):
        hoje = datetime.date.today()
        produtos = []
        for i, (nome, loja, tipo, preco, categoria) in enumerate(self.PRODUTOS_BASE):
            dias = random.choice([1, 1, 2, 2, 3, 4])
            venc = hoje + datetime.timedelta(days=dias)
            desc = random.choice([30, 40, 50, 60])
            preco_desc = round(preco * (1 - desc / 100), 2)
            produtos.append({
                "id": i + 1,
                "nome": nome,
                "loja": loja,
                "tipo": tipo,
                "preco_original": preco,
                "desconto": desc,
                "preco_final": preco_desc,
                "vencimento": venc,
                "categoria": categoria,
                "regiao": random.choice(self.REGIOES),
                "disponivel": True,
                "quantidade": random.randint(1, 8),
            })
        return produtos

    def _criar_missoes(self):
        return [
            {
                "nome": "Resgatador da Semana ",
                "descricao": "Faça 3 resgates esta semana",
                "progresso": 2, "meta": 3,
                "recompensa": 50, "concluida": False,
                "tipo": "semanal",
            },
            {
                "nome": "Mestre das Padarias ",
                "descricao": "Resgate 5 produtos de padarias",
                "progresso": 3, "meta": 5,
                "recompensa": 80, "concluida": False,
                "tipo": "especial",
            },
            {
                "nome": "Herói Regional ",
                "descricao": "Resgate produtos de 3 regiões diferentes",
                "progresso": 1, "meta": 3,
                "recompensa": 120, "concluida": False,
                "tipo": "especial",
            },
            {
                "nome": "Combo Saudável ",
                "descricao": "Resgate 2 produtos vegetais em um dia",
                "progresso": 0, "meta": 2,
                "recompensa": 40, "concluida": False,
                "tipo": "diário",
            },
            {
                "nome": "Estudante Sustentável ",
                "descricao": "Acumule 500 EcoPoints para desbloquear desconto em curso",
                "progresso": 145, "meta": 500,
                "recompensa": 200, "concluida": False,
                "tipo": "educacional",
            },
        ]

    def _criar_historico(self):
        hoje = datetime.date.today()
        historico = []
        produtos_nomes = [p[0] for p in self.PRODUTOS_BASE]
        for i in range(20):
            dias_atras = random.randint(0, 30)
            data = hoje - datetime.timedelta(days=dias_atras)
            nome = random.choice(produtos_nomes)
            historico.append({
                "data": data,
                "produto": nome,
                "pontos": random.randint(10, 60),
                "economia": round(random.uniform(1.5, 8.0), 2),
                "co2_evitado": round(random.uniform(0.1, 0.8), 2),
            })
        return sorted(historico, key=lambda x: x["data"], reverse=True)

    def _criar_doacoes(self):
        return [
            {"ong": "Banco de Alimentos SP", "refeicoes": 342, "parceiros": 8},
            {"ong": "Cozinha Solidária",      "refeicoes": 218, "parceiros": 5},
            {"ong": "Prato Cheio ONG",        "refeicoes": 187, "parceiros": 4},
        ]

    def ranking(self):
        return [
            ("🥇", "Mariana Costa",  2_850, "Zona Norte"),
            ("🥈", "Pedro Alves",    2_410, "Centro"),
            ("🥉", "Juliana Ramos",  2_180, "Zona Sul"),
            ("4°", "Roberto Lima",   1_920, "Zona Leste"),
            ("5°", "Fernanda Cruz",  1_740, "Zona Oeste"),
            ("6°", "Ana Lima",       1_240, "Zona Sul"),
            ("7°", "Lucas Ferreira", 1_100, "Centro"),
            ("8°", "Carlos Melo",      890, "Centro"),
        ]

    def metricas_globais(self):
        return {
            "alimentos_salvos":   8_247,
            "refeicoes_geradas":  12_083,
            "co2_evitado_kg":     4_621,
            "usuarios_ativos":    3_892,
            "parceiros":            148,
            "cidades_atendidas":     24,
            "economia_total_r":  41_380,
        }




def barra_progresso(atual, total, largura=30, cor=Cor.VERDE_CLARO):
    """Renderiza uma barra de progresso colorida."""
    pct = min(atual / total, 1.0) if total > 0 else 0
    preenchido = int(largura * pct)
    vazio = largura - preenchido
    barra = colorir("█" * preenchido, cor) + colorir("░" * vazio, Cor.CINZA)
    return f"[{barra}] {colorir(f'{int(pct*100)}%', Cor.BOLD)}"

def titulo(texto, cor=Cor.VERDE_CLARO, largura=58):
    """Renderiza um cabeçalho formatado."""
    linha = "─" * largura
    print(colorir(f"\n  ┌{linha}┐", Cor.CINZA))
    padding = (largura - len(texto)) // 2
    print(colorir("  │", Cor.CINZA) + " " * padding + colorir(texto, cor + Cor.BOLD) + " " * (largura - padding - len(texto)) + colorir("│", Cor.CINZA))
    print(colorir(f"  └{linha}┘\n", Cor.CINZA))

def caixa(linhas, cor_borda=Cor.CINZA):
    """Renderiza um bloco de conteúdo em caixa."""
    largura = max(len(l) for l in linhas) + 4
    print(colorir("  ┌" + "─" * largura + "┐", cor_borda))
    for linha in linhas:
        pad = largura - len(linha)
        print(colorir("  │ ", cor_borda) + linha + " " * pad + colorir(" │", cor_borda))
    print(colorir("  └" + "─" * largura + "┘", cor_borda))

def animacao_carregando(msg="Carregando", duracao=1.2):
    """Animação de loading humanizada."""
    frames = ["⠋", "⠙", "⠹", "⠸", "⠼", "⠴", "⠦", "⠧", "⠇", "⠏"]
    fim = time.time() + duracao
    i = 0
    while time.time() < fim:
        print(f"\r  {colorir(frames[i % len(frames)], Cor.VERDE_CLARO)}  {colorir(msg + '...', Cor.CINZA)}", end="", flush=True)
        time.sleep(0.1)
        i += 1
    print("\r" + " " * 50 + "\r", end="")

def digitar(texto, delay=0.025, cor=""):
    """Efeito de digitação para textos importantes."""
    for c in texto:
        print(cor + c + Cor.RESET, end="", flush=True)
        time.sleep(delay)
    print()

def separador(char="─", largura=60, cor=Cor.CINZA):
    print(colorir("  " + char * largura, cor))



def tela_boas_vindas():
    limpar()
    logo = f"""
{colorir("  ", Cor.VERDE)}
{colorir("  ", Cor.VERDE)}                                                          {colorir("║", Cor.VERDE)}
{colorir("  ", Cor.VERDE)}   {colorir("  F O O D S A V E   P U L S E  ", Cor.VERDE_CLARO + Cor.BOLD)}              {colorir("║", Cor.VERDE)}
{colorir("  ", Cor.VERDE)}                                                          {colorir("║", Cor.VERDE)}
{colorir("  ", Cor.VERDE)}   {colorir("Combatendo o desperdício alimentar com", Cor.AMARELO)}               {colorir("║", Cor.VERDE)}
{colorir("  ", Cor.VERDE)}   {colorir("tecnologia, sustentabilidade e impacto social.", Cor.AMARELO)}        {colorir("║", Cor.VERDE)}
{colorir("  ", Cor.VERDE)}                                                          {colorir("║", Cor.VERDE)}
{colorir("  ", Cor.VERDE)}
"""
    print(logo)

    stats = [
        (colorir("  8.247", Cor.VERDE_CLARO + Cor.BOLD),   "alimentos salvos"),
        (colorir(" 12.083", Cor.AZUL_CLARO + Cor.BOLD),     "refeições geradas"),
        (colorir("  4.621", Cor.CIANO_CLARO + Cor.BOLD),    "kg CO₂ evitados"),
        (colorir("  3.892", Cor.AMARELO_CLARO + Cor.BOLD),  "usuários ativos"),
    ]

    print(colorir("  ", Cor.CINZA))
    print(colorir("  ", Cor.CINZA) + colorir("    Impacto coletivo desta semana:", Cor.BOLD) + " " * 18 + colorir("│", Cor.CINZA))
    print(colorir("  ", Cor.CINZA) + " " * 54 + colorir("│", Cor.CINZA))
    for valor, label in stats:
        linha = f"  {valor}  {colorir(label, Cor.CINZA)}"
        espaços = 54 - len(f"  {valor}  {label}")
        print(colorir("  ", Cor.CINZA) + linha + " " * max(0, espaços) + colorir("│", Cor.CINZA))
    print(colorir("  ", Cor.CINZA))
    print()


def tela_login(banco):
    titulo("  Entrar na Plataforma", Cor.AZUL_CLARO)

    print(colorir("  Olá! Bem-vindo(a) ao FoodSave Pulse \n", Cor.BRANCO))
    print(colorir("  Contas de demonstração disponíveis:", Cor.CINZA))
    print(colorir("    • Usuário: ", Cor.CINZA) + colorir("ana", Cor.VERDE_CLARO) + colorir("  |  Senha: ", Cor.CINZA) + colorir("1234", Cor.VERDE_CLARO))
    print(colorir("    • Usuário: ", Cor.CINZA) + colorir("carlos", Cor.VERDE_CLARO) + colorir("  |  Senha: ", Cor.CINZA) + colorir("5678", Cor.VERDE_CLARO))
    print()

    usuario = input(colorir("    Usuário: ", Cor.CIANO)).strip().lower()
    senha   = input(colorir("    Senha: ", Cor.CIANO)).strip()

    if usuario in banco.usuarios and banco.usuarios[usuario]["senha"] == senha:
        u = banco.usuarios[usuario]
        banco.usuario_atual = u
        animacao_carregando("Autenticando", 1.0)
        print(colorir(f"\n    Bem-vindo(a) de volta, {u['nome']}! 🎉", Cor.VERDE_CLARO + Cor.BOLD))
        time.sleep(1.0)
        return True
    else:
        print(colorir("\n    Usuário ou senha incorretos. Tente novamente.", Cor.VERMELHO))
        time.sleep(1.5)
        return False


def tela_dashboard(banco):
    limpar()
    u = banco.usuario_atual
    hoje = datetime.date.today()

    titulo(f"  Dashboard  —  Olá, {u['nome'].split()[0]}!", Cor.VERDE_CLARO)


    hora = datetime.datetime.now().hour
    if hora < 12:
        saudacao = "Bom dia"
    elif hora < 18:
        saudacao = "Boa tarde"
    else:
        saudacao = "Boa noite"
    print(f"  {colorir(saudacao + '!', Cor.AMARELO_CLARO + Cor.BOLD)} Hoje é {colorir(hoje.strftime('%d/%m/%Y'), Cor.BRANCO)}.\n")


    nivel_cor = Cor.VERDE_CLARO
    print(colorir("  ", Cor.VERDE))
    print(colorir("  ", Cor.VERDE) + colorir("    Seus EcoPoints     ", Cor.BOLD) + colorir("║", Cor.VERDE) + colorir("  🛒  Total de Resgates  ", Cor.BOLD) + colorir("║", Cor.VERDE))
    print(colorir("  ", Cor.VERDE) + colorir(f"     {u['eco_points']:,} pts".ljust(24), Cor.VERDE_CLARO + Cor.BOLD) + colorir("║", Cor.VERDE) + colorir(f"     {u['resgates']} resgates".ljust(24), Cor.AZUL_CLARO + Cor.BOLD) + colorir("║", Cor.VERDE))
    print(colorir("  ", Cor.VERDE))
    print(colorir("  ", Cor.VERDE) + colorir("    Nível Atual        ", Cor.BOLD) + colorir("║", Cor.VERDE) + colorir("  📍  Sua Região         ", Cor.BOLD) + colorir("║", Cor.VERDE))
    print(colorir("  ", Cor.VERDE) + f"  {colorir(u['nivel'], nivel_cor + Cor.BOLD)}".ljust(36) + colorir("║", Cor.VERDE) + f"  {colorir(u['regiao'], Cor.AMARELO_CLARO)}".ljust(32) + colorir("║", Cor.VERDE))
    print(colorir("  ", Cor.VERDE))


    meta_txt = colorir(f"{u['pontos_mes']}/{u['meta_mensal']} pts", Cor.VERDE_CLARO)
    print(f"\n  {colorir('  Meta do mês:', Cor.BOLD)} {meta_txt}")
    print(f"  {barra_progresso(u['pontos_mes'], u['meta_mensal'])}\n")


    disponiveis = [p for p in banco.produtos if p["disponivel"] and p["vencimento"] >= hoje]
    urgentes = [p for p in disponiveis if (p["vencimento"] - hoje).days <= 1]

    print(f"  {colorir('  Produtos disponíveis:', Cor.BOLD)} {colorir(str(len(disponiveis)), Cor.VERDE_CLARO + Cor.BOLD)}")
    if urgentes:
        print(f"  {colorir(f'    {len(urgentes)} produto(s) vencem hoje ou amanhã!', Cor.AMARELO_CLARO)}")


    print(f"\n  {colorir('  Suas conquistas:', Cor.BOLD)}")
    for c in u["conquistas"]:
        print(f"    {colorir('✓', Cor.VERDE_CLARO)}  {c}")

    separador()
    print(f"\n  {colorir('  Dica do dia:', Cor.AMARELO + Cor.BOLD)} {colorir('Produtos de padaria vencem rápido — confira as ofertas de hoje!', Cor.BRANCO)}\n")


def tela_produtos(banco):
    while True:
        limpar()
        titulo("  Produtos Disponíveis", Cor.VERDE_CLARO)

        hoje = datetime.date.today()
        produtos = [p for p in banco.produtos if p["disponivel"] and p["vencimento"] >= hoje]

        print(f"  {colorir(str(len(produtos)), Cor.VERDE_CLARO + Cor.BOLD)} produtos com desconto disponíveis agora:\n")

        # Filtro
        print(colorir("  Filtrar por: ", Cor.CINZA) + colorir("[1]", Cor.VERDE_CLARO) + " Todos  " + colorir("[2]", Cor.VERDE_CLARO) + " Mercado  " + colorir("[3]", Cor.VERDE_CLARO) + " Restaurante  " + colorir("[4]", Cor.VERDE_CLARO) + " Padaria")
        filtro = input(colorir("  Escolha o filtro (Enter para todos): ", Cor.CIANO)).strip()

        mapa_filtro = {"2": "mercado", "3": "restaurante", "4": "padaria"}
        if filtro in mapa_filtro:
            produtos = [p for p in produtos if p["tipo"] == mapa_filtro[filtro]]

        print()
        print(colorir(f"  {'ID':<4} {'Nome':<22} {'Loja':<25} {'Preço':<8} {'Desc.':<6} {'Vence':<10} {'Região'}", Cor.BOLD))
        separador()

        for p in produtos:
            dias = (p["vencimento"] - hoje).days
            urgencia = colorir("🔴", Cor.VERMELHO) if dias == 0 else colorir("🟡", Cor.AMARELO) if dias == 1 else colorir("🟢", Cor.VERDE_CLARO)
            desc_txt = colorir(f"-{p['desconto']}%", Cor.VERDE_CLARO + Cor.BOLD)
            preco_txt = colorir(f"R${p['preco_final']:.2f}", Cor.VERDE_CLARO)
            venc_txt = p["vencimento"].strftime("%d/%m")
            print(f"  {colorir(str(p['id'])+'.',Cor.CINZA):<4} {urgencia} {p['nome']:<20} {colorir(p['loja'], Cor.AZUL_CLARO):<33} {preco_txt:<17} {desc_txt:<15} {venc_txt:<10} {p['regiao']}")

        separador()
        print(colorir("\n  🔴 Vence hoje  🟡 Vence amanhã  🟢 Mais de 2 dias\n", Cor.CINZA))

        print(colorir("  [número] Resgatar produto   ", Cor.VERDE_CLARO) + colorir("[0] Voltar", Cor.CINZA))
        escolha = input(colorir("\n  Qual produto deseja resgatar? ", Cor.CIANO)).strip()

        if escolha == "0":
            break

        try:
            id_prod = int(escolha)
            produto = next((p for p in produtos if p["id"] == id_prod), None)
            if produto:
                resgatar_produto(banco, produto)
            else:
                print(colorir("\n    Produto não encontrado.", Cor.VERMELHO))
                time.sleep(1.2)
        except ValueError:
            print(colorir("\n    Entrada inválida.", Cor.VERMELHO))
            time.sleep(1.0)


def resgatar_produto(banco, produto):
    limpar()
    u = banco.usuario_atual
    titulo("  Confirmar Resgate", Cor.VERDE_CLARO)

    economia = round(produto["preco_original"] - produto["preco_final"], 2)
    pontos = int(produto["desconto"] * 2)
    co2 = round(random.uniform(0.1, 0.6), 2)

    print(colorir("  Você está prestes a resgatar:\n", Cor.BRANCO))
    caixa([
        f"    {colorir(produto['nome'], Cor.VERDE_CLARO + Cor.BOLD)}",
        f"    {produto['loja']}  —  {produto['regiao']}",
        f"    De {colorir('R$'+str(produto['preco_original']), Cor.CINZA)} por {colorir('R$'+str(produto['preco_final']), Cor.VERDE_CLARO + Cor.BOLD)}  ({colorir(str(produto['desconto'])+'% off', Cor.VERDE_CLARO)})",
        f"    +{colorir(str(pontos)+' EcoPoints', Cor.AMARELO_CLARO + Cor.BOLD)}",
        f"    {colorir(str(co2)+' kg de CO₂ evitado', Cor.CIANO)}",
    ])

    confirmar = input(colorir("\n  Confirmar resgate? [s/N]: ", Cor.CIANO)).strip().lower()
    if confirmar == "s":
        animacao_carregando("Processando resgate", 1.5)

        produto["quantidade"] -= 1
        if produto["quantidade"] == 0:
            produto["disponivel"] = False

        u["eco_points"] += pontos
        u["resgates"]   += 1
        u["pontos_mes"] += pontos

        print(colorir("\n    Resgate realizado com sucesso!\n", Cor.VERDE_CLARO + Cor.BOLD))
        digitar(f"  Você ganhou +{pontos} EcoPoints! Seu total: {u['eco_points']:,} pts", delay=0.02, cor=Cor.AMARELO_CLARO)
        digitar(f"  {co2} kg de CO₂ deixarão de ser emitidos. Obrigado! ", delay=0.02, cor=Cor.CIANO)


        if u["eco_points"] >= 2000 and u["nivel"] != "Campeão da Terra ":
            print(colorir("\n    PARABÉNS! Você subiu para o nível ", Cor.AMARELO_CLARO + Cor.BOLD) + colorir("Campeão da Terra 🏆", Cor.VERDE_CLARO + Cor.BOLD))
            u["nivel"] = "Campeão da Terra "

        pausar()
    else:
        print(colorir("\n  Resgate cancelado.", Cor.CINZA))
        time.sleep(1.0)


def tela_missoes(banco):
    limpar()
    u = banco.usuario_atual
    titulo("  Missões e Desafios", Cor.AMARELO_CLARO)

    print(f"  Complete missões e ganhe {colorir('EcoPoints extras', Cor.VERDE_CLARO + Cor.BOLD)}!\n")

    tipos_cor = {
        "diário":       Cor.AZUL_CLARO,
        "semanal":      Cor.VERDE_CLARO,
        "especial":     Cor.AMARELO_CLARO,
        "educacional":  Cor.MAGENTA,
    }

    for i, m in enumerate(banco.missoes, 1):
        cor = tipos_cor.get(m["tipo"], Cor.BRANCO)
        status = colorir(" CONCLUÍDA", Cor.VERDE_CLARO + Cor.BOLD) if m["concluida"] else colorir("⏳ Em andamento", Cor.CINZA)
        tipo_badge = f"[{m['tipo'].upper()}]"

        print(f"  {colorir(str(i)+'.', Cor.CINZA)}  {colorir(m['nome'], Cor.BOLD)}  {colorir(tipo_badge, cor)}")
        print(f"      {colorir(m['descricao'], Cor.CINZA)}")
        prog_txt = colorir(f"{m['progresso']}/{m['meta']}", Cor.BRANCO)
        print(f"      {barra_progresso(m['progresso'], m['meta'], largura=25)}  {prog_txt}")
        print(f"      {colorir(' Recompensa:', Cor.CINZA)} {colorir(str(m['recompensa'])+' EcoPoints', Cor.VERDE_CLARO + Cor.BOLD)}  |  {status}")
        print()

    separador()
    print(f"\n  {colorir('  Benefício Educacional:', Cor.MAGENTA + Cor.BOLD)}")
    print(f"  Com {colorir('500+ EcoPoints', Cor.VERDE_CLARO + Cor.BOLD)}, você desbloqueia:")
    print(f"    • {colorir('Desconto de 20% em cursos EAD parceiros', Cor.BRANCO)}")
    print(f"    • {colorir('Bolsas parciais em instituições conveniadas', Cor.BRANCO)}")
    print(f"    • {colorir('Redução de mensalidades (nível Campeão)', Cor.BRANCO)}")
    print(f"\n  {colorir('Seu status:', Cor.CINZA)} {colorir(str(u['eco_points'])+' pts', Cor.VERDE_CLARO + Cor.BOLD)}  {barra_progresso(u['eco_points'], 500, largura=20)}\n")

    pausar()


def tela_ranking(banco):
    limpar()
    titulo("  Ranking Sustentável", Cor.AMARELO_CLARO)

    print(f"  {colorir('Os maiores heróis do combate ao desperdício:', Cor.BRANCO)}\n")

    u = banco.usuario_atual
    ranking = banco.ranking()

    medalhas_cor = {
        "🥇": Cor.AMARELO_CLARO,
        "🥈": Cor.CINZA,
        "🥉": Cor.AMARELO,
    }

    print(colorir(f"  {'Pos.':<5} {'Nome':<22} {'EcoPoints':<14} {'Região'}", Cor.BOLD + Cor.CINZA))
    separador()

    for pos, nome, pts, regiao in ranking:
        destaque = nome == u["nome"]
        cor_nome = Cor.VERDE_CLARO + Cor.BOLD if destaque else Cor.BRANCO
        cor_pos  = medalhas_cor.get(pos, Cor.CINZA)
        seta = colorir("  ← você!", Cor.VERDE_CLARO) if destaque else ""
        pts_fmt = colorir(f"{pts:,} pts", Cor.VERDE_CLARO if destaque else Cor.CINZA)

        print(f"  {colorir(pos, cor_pos):<8} {colorir(nome, cor_nome):<30} {pts_fmt:<22} {regiao}{seta}")

    separador()
    print(f"\n  {colorir('  Impacto coletivo do TOP 10 esta semana:', Cor.BOLD)}")
    print(f"      {colorir('284 refeições geradas', Cor.VERDE_CLARO)}")
    print(f"      {colorir('127 kg de CO₂ evitados', Cor.CIANO)}")
    print(f"      {colorir('R$ 1.342 economizados', Cor.AMARELO_CLARO)}\n")

    pausar()


def tela_historico(banco):
    limpar()
    titulo("  Meu Histórico de Resgates", Cor.AZUL_CLARO)

    historico = banco.historico[:10]
    total_pts   = sum(h["pontos"] for h in banco.historico)
    total_eco   = round(sum(h["economia"] for h in banco.historico), 2)
    total_co2   = round(sum(h["co2_evitado"] for h in banco.historico), 2)

    print(colorir("  Resumo geral:\n", Cor.BOLD))
    print(f"     EcoPoints acumulados: {colorir(str(total_pts), Cor.VERDE_CLARO + Cor.BOLD)}")
    print(f"     Economia total:       {colorir('R$ '+str(total_eco), Cor.AMARELO_CLARO + Cor.BOLD)}")
    print(f"     CO₂ evitado:          {colorir(str(total_co2)+' kg', Cor.CIANO + Cor.BOLD)}")

    separador()
    print(colorir(f"\n  Últimos 10 resgates:\n", Cor.BOLD))
    print(colorir(f"  {'Data':<12} {'Produto':<22} {'Pontos':<10} {'Economia':<12} {'CO₂ evitado'}", Cor.CINZA))
    separador()

    for h in historico:
        data_str  = h["data"].strftime("%d/%m/%Y")
        eco_str   = colorir(f"R$ {h['economia']:.2f}", Cor.AMARELO_CLARO)
        pts_str   = colorir(f"+{h['pontos']} pts", Cor.VERDE_CLARO)
        co2_str   = colorir(f"{h['co2_evitado']:.2f} kg", Cor.CIANO)
        print(f"  {data_str:<12} {h['produto']:<22} {pts_str:<19} {eco_str:<20} {co2_str}")

    print()
    pausar()


def tela_mapa_desperdicio(banco):
    limpar()
    titulo("🗺️  Mapa de Desperdício Urbano", Cor.CIANO_CLARO)

    print(f"  {colorir('Análise territorial de desperdício alimentar', Cor.BRANCO)}\n")
    print(f"  {colorir('Como funciona:', Cor.BOLD)} Nossa IA analisa dados de geolocalização,")
    print(f"  densidade populacional e histórico de resgates para")
    print(f"  identificar regiões que mais precisam de atenção.\n")

    dados_regioes = [
        ("Centro",     82, 143, "Alta demanda e alta oferta",  "★★★★★"),
        ("Zona Norte", 65, 97,  "Crescimento acelerado",       "★★★★"),
        ("Zona Sul",   91, 78,  "Maior desperdício per capita","★★★"),
        ("Zona Leste", 44, 112, "Suprimento insuficiente",     "★★★★"),
        ("Zona Oeste", 58, 89,  "Equilíbrio em melhora",       "★★★"),
    ]

    print(colorir(f"  {'Região':<14} {'Oferta':<8} {'Demanda':<10} {'Status':<30} {'Score'}", Cor.BOLD + Cor.CINZA))
    separador()

    for regiao, oferta, demanda, status, score in dados_regioes:
        destaque = regiao == banco.usuario_atual["regiao"]
        cor = Cor.VERDE_CLARO + Cor.BOLD if destaque else Cor.BRANCO
        seta = colorir("  ← você!", Cor.VERDE_CLARO) if destaque else ""
        bal = oferta - demanda
        bal_txt = colorir(f"(+{bal})", Cor.VERDE_CLARO) if bal >= 0 else colorir(f"({bal})", Cor.VERMELHO)
        print(f"  {colorir(regiao, cor):<22} {colorir(str(oferta)+'%', Cor.AZUL_CLARO):<16} {colorir(str(demanda)+'%', Cor.AMARELO):<18} {status:<30} {score}{seta}")

    separador()
    print(f"\n  {colorir('  Missões Regionais Ativas:', Cor.BOLD + Cor.AMARELO_CLARO)}")
    print(f"    • {colorir('Zona Sul', Cor.VERMELHO + Cor.BOLD)}: \"Precisa de heróis!\" — bônus de {colorir('2x EcoPoints', Cor.VERDE_CLARO)}")
    print(f"    • {colorir('Zona Leste', Cor.AMARELO + Cor.BOLD)}: \"Conectar oferta-demanda\" — {colorir('+100 pts ao completar', Cor.VERDE_CLARO)}\n")

    print(f"  {colorir('  Dica:', Cor.CIANO + Cor.BOLD)} Resgate produtos na Zona Sul esta semana")
    print(f"  e ganhe o {colorir('dobro de EcoPoints', Cor.VERDE_CLARO + Cor.BOLD)} + badge regional!\n")

    pausar()


def tela_impacto_ambiental(banco):
    limpar()
    titulo("  Impacto Ambiental & Social", Cor.CIANO_CLARO)

    metricas = banco.metricas_globais()
    u = banco.usuario_atual

    print(colorir("    Métricas globais da plataforma em tempo real:\n", Cor.BOLD))

    cards = [
        ("  Alimentos salvos",    f"{metricas['alimentos_salvos']:,}",     "unidades",    Cor.VERDE_CLARO),
        ("  Refeições geradas",   f"{metricas['refeicoes_geradas']:,}",    "refeições",   Cor.AZUL_CLARO),
        ("  CO₂ evitado",        f"{metricas['co2_evitado_kg']:,} kg",    "emissões",    Cor.CIANO_CLARO),
        ("  Economia gerada",    f"R$ {metricas['economia_total_r']:,}",  "para usuários",Cor.AMARELO_CLARO),
        ("  Usuários ativos",     f"{metricas['usuarios_ativos']:,}",      "pessoas",     Cor.MAGENTA),
        ("  Parceiros",           f"{metricas['parceiros']:,}",            "empresas",    Cor.VERDE_CLARO),
    ]

    for icone_nome, valor, unidade, cor in cards:
        bar = colorir("█" * 30, cor)
        print(f"  {colorir(icone_nome+':', Cor.BOLD):<30}  {colorir(valor, cor + Cor.BOLD):>12}  {colorir(unidade, Cor.CINZA)}")
        print()

    separador()
    print(colorir("\n    Doações para ONGs Parceiras:\n", Cor.BOLD))
    for d in banco.doacoes:
        print(f"    • {colorir(d['ong'], Cor.VERDE_CLARO + Cor.BOLD)}: {colorir(str(d['refeicoes'])+' refeições', Cor.BRANCO)} distribuídas via {d['parceiros']} parceiros")

    separador()
    print(f"\n  {colorir('  Sua contribuição pessoal, {0}:'.format(u['nome'].split()[0]), Cor.BOLD)}")
    print(f"     {colorir(str(u['eco_points'])+' EcoPoints', Cor.VERDE_CLARO + Cor.BOLD)} acumulados")
    print(f"     {colorir(str(u['resgates'])+' alimentos salvos', Cor.AZUL_CLARO)} do descarte")
    print(f"     Equivalente a {colorir(str(round(u['resgates'] * 0.35, 1))+' kg de CO₂ evitados', Cor.CIANO)}\n")

    pausar()


def tela_buscar_produto(banco):
    """Permite ao usuário buscar produtos por nome ou categoria.
    Utiliza strings, listas, for, if/elif/else e retorna resultado formatado.
    """
    limpar()
    titulo("  Buscar Produto", Cor.AZUL_CLARO)

    print(colorir("  Opções de busca:", Cor.BOLD))
    print(colorir("  [1]", Cor.VERDE_CLARO) + " Buscar por nome")
    print(colorir("  [2]", Cor.VERDE_CLARO) + " Buscar por categoria")
    print(colorir("  [0]", Cor.CINZA) + " Voltar\n")

    opcao = input(colorir("  Escolha: ", Cor.CIANO)).strip()

    if opcao == "0":
        return
    elif opcao not in ("1", "2"):
        print(colorir("\n  Opção inválida.", Cor.VERMELHO))
        time.sleep(1.0)
        return

    hoje = datetime.date.today()
    produtos_disponiveis = [p for p in banco.produtos if p["disponivel"] and p["vencimento"] >= hoje]

    # Coleta e valida o termo de busca
    if opcao == "1":
        termo = input(colorir("  Digite o nome do produto: ", Cor.CIANO)).strip()
        if len(termo) < 2:
            print(colorir("\n  Termo muito curto. Digite ao menos 2 caracteres.", Cor.VERMELHO))
            time.sleep(1.2)
            return
        resultados = [p for p in produtos_disponiveis if termo.lower() in p["nome"].lower()]
    else:
        categorias = list({p["categoria"] for p in produtos_disponiveis})
        print(colorir("\n  Categorias disponíveis:", Cor.BOLD))
        for i, cat in enumerate(categorias, 1):
            print(f"    {colorir(str(i)+'.', Cor.CINZA)} {cat}")
        cat_escolha = input(colorir("\n  Categoria: ", Cor.CIANO)).strip().lower()
        resultados = [p for p in produtos_disponiveis if cat_escolha in p["categoria"].lower()]

    separador()

    # Exibe resultados ou aviso de lista vazia
    if not resultados:
        print(colorir("\n  Nenhum produto encontrado para sua busca.", Cor.AMARELO))
    else:
        print(colorir(f"\n  {len(resultados)} produto(s) encontrado(s):\n", Cor.VERDE_CLARO + Cor.BOLD))
        print(colorir(f"  {'Nome':<22} {'Loja':<25} {'Preço':<10} {'Desc.':<8} {'Vence'}", Cor.BOLD))
        separador()
        for p in resultados:
            preco_txt = colorir(f"R${p['preco_final']:.2f}", Cor.VERDE_CLARO)
            desc_txt  = colorir(f"-{p['desconto']}%", Cor.AMARELO_CLARO)
            venc_txt  = p["vencimento"].strftime("%d/%m")
            print(f"  {p['nome']:<22} {colorir(p['loja'], Cor.AZUL_CLARO):<33} {preco_txt:<17} {desc_txt:<15} {venc_txt}")

    print()
    pausar()


def tela_cadastrar_usuario(banco):
    """Cadastra um novo usuário na plataforma com validação de dados.
    Demonstra: input(), print(), if/elif/else, while, strings, listas, tuplas e funções com retorno.
    """
    limpar()
    titulo("  Cadastrar Novo Usuário", Cor.VERDE_CLARO)

    print(colorir("  Preencha os dados abaixo para criar sua conta:\n", Cor.BRANCO))

    # Tupla com regiões válidas (imutável por design)
    regioes_validas = ("Centro", "Zona Norte", "Zona Sul", "Zona Leste", "Zona Oeste")

    # --- Nome de usuário ---
    while True:
        usuario = input(colorir("  Nome de usuário (sem espaços, mín. 3 letras): ", Cor.CIANO)).strip().lower()
        if len(usuario) < 3:
            print(colorir("  Usuário deve ter ao menos 3 caracteres.", Cor.VERMELHO))
        elif " " in usuario:
            print(colorir("  Não use espaços no nome de usuário.", Cor.VERMELHO))
        elif usuario in banco.usuarios:
            print(colorir("  Esse usuário já existe. Escolha outro.", Cor.VERMELHO))
        else:
            break

    # --- Nome completo ---
    while True:
        nome = input(colorir("  Nome completo: ", Cor.CIANO)).strip()
        if len(nome) < 5 or " " not in nome:
            print(colorir("  Informe nome e sobrenome.", Cor.VERMELHO))
        else:
            break

    # --- E-mail ---
    while True:
        email = input(colorir("  E-mail: ", Cor.CIANO)).strip()
        if "@" not in email or "." not in email:
            print(colorir("  E-mail inválido. Ex: usuario@email.com", Cor.VERMELHO))
        else:
            break

    # --- Senha ---
    while True:
        senha = input(colorir("  Senha (mín. 4 caracteres): ", Cor.CIANO)).strip()
        if len(senha) < 4:
            print(colorir("  Senha muito curta.", Cor.VERMELHO))
        else:
            break

    # --- Região ---
    print(colorir("\n  Regiões disponíveis:", Cor.BOLD))
    for i, r in enumerate(regioes_validas, 1):
        print(f"    {colorir(str(i)+'.', Cor.CINZA)} {r}")

    while True:
        escolha_regiao = input(colorir("  Escolha o número da sua região: ", Cor.CIANO)).strip()
        if escolha_regiao.isdigit() and 1 <= int(escolha_regiao) <= len(regioes_validas):
            regiao = regioes_validas[int(escolha_regiao) - 1]
            break
        else:
            print(colorir("  Opção inválida. Escolha entre 1 e 5.", Cor.VERMELHO))

    # Cria o novo usuário e adiciona ao banco em memória
    novo_usuario = {
        "nome": nome,
        "email": email,
        "senha": senha,
        "eco_points": 0,
        "resgates": 0,
        "nivel": "Iniciante Verde ",
        "regiao": regiao,
        "tipo": "consumidor",
        "historico_pontos": [0] * 12,
        "conquistas": [],
        "meta_mensal": 100,
        "pontos_mes": 0,
    }
    banco.usuarios[usuario] = novo_usuario

    animacao_carregando("Criando conta", 1.2)
    print(colorir(f"\n  Conta criada com sucesso! Bem-vindo(a), {nome.split()[0]}! 🎉", Cor.VERDE_CLARO + Cor.BOLD))
    print(colorir(f"  Use '{usuario}' e sua senha para entrar na plataforma.\n", Cor.CINZA))
    pausar()


def tela_relatorio(banco):
    """Gera um relatório textual completo da sessão do usuário logado.
    Utiliza: for, listas, tuplas, strings, if/elif/else, funções com retorno.
    """
    limpar()
    titulo("  Relatório da Sessão", Cor.MAGENTA)

    u = banco.usuario_atual
    hoje = datetime.date.today()

    # --- Cabeçalho ---
    print(colorir(f"  Usuário:  ", Cor.BOLD) + colorir(u["nome"], Cor.VERDE_CLARO + Cor.BOLD))
    print(colorir(f"  Região:   ", Cor.BOLD) + u["regiao"])
    print(colorir(f"  Nível:    ", Cor.BOLD) + colorir(u["nivel"], Cor.AMARELO_CLARO))
    print(colorir(f"  Data:     ", Cor.BOLD) + hoje.strftime("%d/%m/%Y"))
    separador()

    # --- Pontuação e metas ---
    print(colorir("\n  Pontuação & Metas\n", Cor.BOLD))
    pct_meta = round((u["pontos_mes"] / u["meta_mensal"]) * 100, 1) if u["meta_mensal"] > 0 else 0
    print(f"  Total de EcoPoints:  {colorir(str(u['eco_points']), Cor.VERDE_CLARO + Cor.BOLD)}")
    print(f"  Pontos este mês:     {colorir(str(u['pontos_mes']), Cor.VERDE_CLARO)} / {u['meta_mensal']}  ({colorir(str(pct_meta)+'%', Cor.AMARELO_CLARO)} da meta)")
    print(f"  Total de resgates:   {colorir(str(u['resgates']), Cor.AZUL_CLARO)}")

    # Feedback sobre meta mensal com if/elif/else
    if pct_meta >= 100:
        feedback_meta = colorir("  Meta atingida! Parabéns!", Cor.VERDE_CLARO + Cor.BOLD)
    elif pct_meta >= 50:
        feedback_meta = colorir("  Você está na metade do caminho. Continue!", Cor.AMARELO)
    else:
        feedback_meta = colorir("  Você ainda tem muito a conquistar este mês!", Cor.VERMELHO)
    print(f"  {feedback_meta}\n")

    separador()

    # --- Missões ---
    print(colorir("\n  Status das Missões\n", Cor.BOLD))
    missoes_concluidas = [m for m in banco.missoes if m["concluida"]]
    missoes_pendentes  = [m for m in banco.missoes if not m["concluida"]]

    print(f"  Concluídas: {colorir(str(len(missoes_concluidas)), Cor.VERDE_CLARO + Cor.BOLD)}  |  Pendentes: {colorir(str(len(missoes_pendentes)), Cor.AMARELO)}")

    for m in banco.missoes:
        status_txt = colorir("✓ CONCLUÍDA", Cor.VERDE_CLARO) if m["concluida"] else colorir(f"{m['progresso']}/{m['meta']}", Cor.CINZA)
        print(f"    • {m['nome']:<35} {status_txt}")

    separador()

    # --- Conquistas ---
    print(colorir("\n  Conquistas Desbloqueadas\n", Cor.BOLD))
    if u["conquistas"]:
        for c in u["conquistas"]:
            print(f"    {colorir('✓', Cor.VERDE_CLARO)}  {c}")
    else:
        print(colorir("    Nenhuma conquista ainda. Faça seu primeiro resgate!", Cor.CINZA))

    separador()

    # --- Histórico resumido (últimos 5) ---
    print(colorir("\n  Últimos 5 Resgates\n", Cor.BOLD))
    ultimos = banco.historico[:5]
    if ultimos:
        for h in ultimos:
            print(f"    {h['data'].strftime('%d/%m')}  {h['produto']:<22}  +{colorir(str(h['pontos'])+' pts', Cor.VERDE_CLARO)}  R${h['economia']:.2f} economizados")
    else:
        print(colorir("    Nenhum resgate registrado.", Cor.CINZA))

    # --- Impacto pessoal estimado ---
    separador()
    print(colorir("\n  Impacto Pessoal Estimado\n", Cor.BOLD))
    co2_pessoal  = round(u["resgates"] * 0.35, 2)
    eco_pessoal  = round(u["resgates"] * 3.50, 2)
    print(f"  CO₂ evitado:     {colorir(str(co2_pessoal)+' kg', Cor.CIANO + Cor.BOLD)}")
    print(f"  Economia gerada: {colorir('R$ '+str(eco_pessoal), Cor.AMARELO_CLARO + Cor.BOLD)}\n")

    pausar()


def tela_sobre(banco):
    limpar()
    titulo("ℹ   Sobre o FoodSave Pulse", Cor.VERDE_CLARO)

    sobre = [
        ("  Missão",     "Transformar desperdício em oportunidade por meio da tecnologia."),
        ("  Parceiros",  "Mercados, restaurantes, padarias, ONGs e instituições de ensino."),
        ("  Gamificação","EcoPoints, rankings, missões e desafios sustentáveis."),
        ("  Educação",   "Descontos em cursos EAD e bolsas via pontos acumulados."),
        ("  Tecnologia", "Geolocalização, IA preditiva e cidades inteligentes."),
        ("  Modelo",     "Parcerias, assinaturas premium e campanhas patrocinadas."),
    ]

    for icone_cat, descricao in sobre:
        print(f"  {colorir(icone_cat + ':', Cor.VERDE_CLARO + Cor.BOLD)}")
        print(f"    {colorir(descricao, Cor.BRANCO)}\n")

    separador()
    print(colorir("\n    Tecnologias utilizadas:\n", Cor.BOLD))
    techs = ["Python 3.12", "Matplotlib", "Geolocalização GPS", "IA preditiva", "Cloud Computing", "API REST"]
    for t in techs:
        print(f"    {colorir('▸', Cor.VERDE_CLARO)}  {t}")

    print(colorir("\n\n  © 2025 FoodSave Pulse — Todos os direitos reservados 🌍\n", Cor.CINZA))
    pausar()




def menu_principal(banco):
    """Menu principal do sistema com pelo menos 5 opções funcionais.
    Utiliza: while, for, if/elif/else, listas, tuplas e funções.
    """
    # Lista de tuplas com (tecla, label, função) — uso de tuplas e lista
    opcoes = [
        ("1", "  Ver Produtos Disponíveis",   tela_produtos),
        ("2", "  Buscar Produto",             tela_buscar_produto),
        ("3", "  Missões e Desafios",          tela_missoes),
        ("4", "  Ranking Sustentável",         tela_ranking),
        ("5", "  Meu Histórico",              tela_historico),
        ("6", "   Mapa de Desperdício",         tela_mapa_desperdicio),
        ("7", "  Impacto Ambiental & Social",  tela_impacto_ambiental),
        ("8", "  Relatório da Sessão",         tela_relatorio),
        ("9", "   Sobre o Projeto",             tela_sobre),
        ("0", "  Sair",                        None),
    ]

    while True:
        tela_dashboard(banco)

        print(colorir("  ┌─────────────────────────────────────────────────┐", Cor.CINZA))
        print(colorir("  │", Cor.CINZA) + colorir("    O que você quer fazer hoje?", Cor.BOLD) + " " * 18 + colorir("│", Cor.CINZA))
        print(colorir("  │", Cor.CINZA) + " " * 51 + colorir("│", Cor.CINZA))
        for key, label, _ in opcoes:
            pad = 49 - len(f"  [{key}]  {label}")
            print(colorir("  │", Cor.CINZA) + f"  {colorir('['+key+']', Cor.VERDE_CLARO)}  {label}" + " " * max(0, pad) + colorir("│", Cor.CINZA))
        print(colorir("  └─────────────────────────────────────────────────┘\n", Cor.CINZA))

        escolha = input(colorir("  Sua escolha: ", Cor.CIANO + Cor.BOLD)).strip()

        if escolha == "0":
            limpar()
            print(colorir("\n  Até a próxima! Continuem salvando o planeta, juntos! \n", Cor.VERDE_CLARO + Cor.BOLD))
            time.sleep(1.5)
            break

        # Percorre a lista de opções com for e valida com if/else
        encontrou = False
        for key, _, func in opcoes:
            if escolha == key and func:
                func(banco)
                encontrou = True
                break
        if not encontrou and escolha != "0":
            print(colorir("\n  Opção inválida. Tente novamente.", Cor.VERMELHO))
            time.sleep(1.0)




def main():
    """Função principal: inicializa o banco e exibe a tela de boas-vindas com login ou cadastro."""
    banco = Banco()

    while True:
        tela_boas_vindas()

        print(colorir("  [1]", Cor.VERDE_CLARO) + " Entrar na plataforma")
        print(colorir("  [2]", Cor.AZUL_CLARO)  + " Criar nova conta")
        print(colorir("  [0]", Cor.CINZA)        + " Sair")
        print()
        acao = input(colorir("  Escolha: ", Cor.CIANO)).strip()

        if acao == "0":
            print(colorir("\n  Até mais! \n", Cor.VERDE_CLARO))
            break
        elif acao == "1":
            sucesso = tela_login(banco)
            if sucesso:
                menu_principal(banco)
                banco.usuario_atual = None
        elif acao == "2":
            tela_cadastrar_usuario(banco)
        else:
            print(colorir("\n  Opção inválida.", Cor.VERMELHO))
            time.sleep(1.0)


if __name__ == "__main__":
    main(),


  
                                                            ║
       F O O D S A V E   P U L S E                ║
                                                            ║
     Combatendo o desperdício alimentar com               ║
     tecnologia, sustentabilidade e impacto social.        ║
                                                            ║
  

  
      Impacto coletivo desta semana:                  │
                                                        │
      8.247  alimentos salvos              │
     12.083  refeições geradas             │
      4.621  kg CO₂ evitados               │
      3.892  usuários ativos               │
  

  [1] Entrar na plataforma
  [2] Criar nova conta
  [0] Sair

